# HotpotQA — BM25 + ColBERT + RRF

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/Datasets/hotpotqa"
TOP_K = 100
RANDOM_SEED = 42
NON_RELEVANT_PAD = 35000

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q "numpy==1.26.4" --force-reinstall
!pip install -q beir pytrec-eval-terrier

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version

# Restart Session and then Run Next Cell

In [ ]:
%%time
from beir.datasets.data_loader import GenericDataLoader
import re, unicodedata, random

DATA_PATH = "/content/drive/MyDrive/Datasets/hotpotqa"
TOP_K = 100
RANDOM_SEED = 42
NON_RELEVANT_PAD = 35000

corpus, queries, qrels = GenericDataLoader(data_folder=DATA_PATH).load(split="test")
print(f"Full corpus: {len(corpus):,} docs")
print(f"Queries: {len(queries):,}")
print(f"Qrels: {len(qrels):,}")

def shared_clean(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for doc_id, doc in corpus.items():
    doc["title"] = shared_clean(doc.get("title", ""))
    doc["text"]  = shared_clean(doc.get("text", ""))

for q_id, q in queries.items():
    queries[q_id] = shared_clean(q)

# --- relevance-preserving subset ---
relevant_doc_ids = set()
for q_id, docs in qrels.items():
    relevant_doc_ids.update(docs.keys())

corpus_subset = {did: corpus[did] for did in relevant_doc_ids if did in corpus}
n_relevant = len(corpus_subset)
print(f"Relevant docs in corpus: {n_relevant:,}")

non_relevant_ids = [did for did in corpus if did not in relevant_doc_ids]
random.seed(RANDOM_SEED)
pad_ids = random.sample(non_relevant_ids, min(NON_RELEVANT_PAD, len(non_relevant_ids)))
for did in pad_ids:
    corpus_subset[did] = corpus[did]

subset_doc_ids = list(corpus_subset.keys())
SUBSET_SIZE = len(corpus_subset)
print(f"Final subset: {SUBSET_SIZE:,} docs ({n_relevant:,} relevant + {SUBSET_SIZE - n_relevant:,} non-relevant)")

/usr/local/lib/python3.12/dist-packages/beir/datasets/data_loader.py:8: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/5233329 [00:00<?, ?it/s]

Full corpus: 5,233,329 docs
Queries: 7,405
Qrels: 7,405
Relevant docs in corpus: 13,783
Final subset: 48,783 docs (13,783 relevant + 35,000 non-relevant)
CPU times: user 2min 27s, sys: 5.02 s, total: 2min 32s
Wall time: 2min 53s


## BM25 Retrieval

In [ ]:
!pip install -q elasticsearch==7.9.1
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-oss-7.9.2-linux-x86_64.tar.gz.sha512
!tar -xzf elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!sudo chown -R daemon:daemon elasticsearch-7.9.2/
!shasum -a 512 -c elasticsearch-oss-7.9.2-linux-x86_64.tar.gz.sha512

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.2/219.2 kB 7.7 MB/s eta 0:00:00
elasticsearch-oss-7.9.2-linux-x86_64.tar.gz: OK


In [ ]:
%%bash --bg
sudo -H -u daemon elasticsearch-7.9.2/bin/elasticsearch

In [ ]:
import time
time.sleep(30)
!curl -s http://localhost:9200 | python3 -c 'import sys, json; d=json.load(sys.stdin); print(f"ES {d[\"version\"][\"number\"]} is up")'

  File "<string>", line 1
    import sys, json; d=json.load(sys.stdin); print(f"ES {d[\"version\"][\"number\"]} is up")
                                                             ^
SyntaxError: unexpected character after line continuation character


In [ ]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

es = Elasticsearch("http://localhost:9200")
assert es.ping(), "ES not reachable!"

bm25_index_name = "hotpotqa_bm25"

bm25_index_config = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "similarity": {
            "default": {
                "type": "BM25",
                "k1": 0.9,
                "b": 0.4
            }
        }
    },
    "mappings": {
        "properties": {
            "title": {"type": "text"},
            "text": {"type": "text"}
        }
    }
}

if es.indices.exists(index=bm25_index_name):
    es.indices.delete(index=bm25_index_name)
    print("Old index deleted")

es.indices.create(index=bm25_index_name, body=bm25_index_config)
print("Index created")

# Bulk index
def generate_bm25_docs():
    for doc_id, doc in corpus_subset.items():
        yield {
            "_index": bm25_index_name,
            "_id": doc_id,
            "_source": {
                "title": doc.get("title", ""),
                "text": doc.get("text", "")
            }
        }

success, failed = bulk(
    es,
    generate_bm25_docs(),
    chunk_size=500,
    request_timeout=300,
    stats_only=True,
    refresh='wait_for'
)
print(f"Indexed: {success:,} success, {failed} failed")

es.indices.refresh(index=bm25_index_name)

count = es.count(index=bm25_index_name)["count"]
print(f"Indexed count after refresh: {count:,}")

assert count == success, f"Expected {success}, got {count}"
print(f"Verified: {count:,} docs in index")

Old index deleted
Index created
Indexed: 48,783 success, 0 failed
Indexed count after refresh: 48,783
Verified: 48,783 docs in index


In [ ]:
sample_q = list(queries.values())[0]
res = es.search(index=bm25_index_name, body={"query": {"multi_match": {"query": sample_q, "fields": ["title","text"]}}, "size": 3})
print(f"Query: {sample_q}")
for hit in res["hits"]["hits"]:
    print(f"  [{hit['_score']:.2f}] {hit['_source']['title'][:80]}")

Query: Were Scott Derrickson and Ed Wood of the same nationality?
  [17.95] Scott Derrickson
  [16.79] That Darn Cat (1997 film)
  [16.74] The Right Stuff (film)


In [ ]:
%%time
from tqdm import tqdm

bm25_results = {}
for q_id, q_text in tqdm(queries.items(), desc="BM25"):
    try:
        res = es.search(
            index=bm25_index_name,
            body={"query": {"multi_match": {"query": q_text, "fields": ["title","text"]}}, "size": TOP_K},
            request_timeout=30
        )
        bm25_results[q_id] = {hit["_id"]: float(hit["_score"]) for hit in res["hits"]["hits"]}
    except Exception as e:
        print(f"Error {q_id}: {e}")
        bm25_results[q_id] = {}

print(f"BM25 retrieval done: {len(bm25_results):,} queries")

BM25: 100%|██████████| 7405/7405 [02:15<00:00, 54.80it/s]

BM25 retrieval done: 7,405 queries
CPU times: user 9.8 s, sys: 749 ms, total: 10.5 s
Wall time: 2min 15s


In [ ]:
import pytrec_eval, pickle

qrels_int = {q: {d: int(s) for d, s in docs.items()} for q, docs in qrels.items()}
metrics = {"ndcg_cut.10", "recip_rank.10", "recall.100"}
evaluator = pytrec_eval.RelevanceEvaluator(qrels_int, metrics)

bm25_eval = evaluator.evaluate(bm25_results)
bm25_ndcg   = [bm25_eval[q]["ndcg_cut_10"] for q in bm25_eval]
bm25_mrr    = [bm25_eval[q]["recip_rank_10"]   for q in bm25_eval]
bm25_recall = [bm25_eval[q]["recall_100"]   for q in bm25_eval]

print("=" * 50)
print("BM25 Results — HotpotQA")
print("=" * 50)
print(f"nDCG@10:    {sum(bm25_ndcg)/len(bm25_ndcg):.4f}")
print(f"MRR:        {sum(bm25_mrr)/len(bm25_mrr):.4f}")
print(f"Recall@100: {sum(bm25_recall)/len(bm25_recall):.4f}")

with open("/content/drive/MyDrive/IR CW2/bm25_results_hotpotqa.pkl", "wb") as f:
    pickle.dump(bm25_results, f)
print("BM25 results saved to Drive")

BM25 Results — HotpotQA
nDCG@10:    0.8120
MRR:        0.9356
Recall@100: 0.9211
BM25 results saved to Drive


In [ ]:
!pip uninstall -y numpy transformers ragatouille langchain langchain-core colbert-ai -q
!pip install -q numpy==1.26.4
!pip install -q transformers==4.37.2
!pip install -q ragatouille==0.0.8
!pip install -q langchain==0.1.20 langchain-core==0.1.53
!pip install -q beir pytrec-eval-terrier
print("ColBERT packages installed — RESTART SESSION NOW, then run from next cell")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.1 requires transformers, which is not installed.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, which is not installed.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.

In [ ]:
import numpy as np
import torch
print(f"numpy: {np.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import transformers
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
transformers.AdamW = AdamW
transformers.get_linear_schedule_with_warmup = get_linear_schedule_with_warmup

from ragatouille import RAGPretrainedModel
print("RAGatouille loaded")

import time as time_module
import ragatouille.models.index as rag_index
rag_index.time = time_module
print("Patches applied")

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


numpy: 1.26.4
CUDA: True
GPU: Tesla T4
RAGatouille loaded
Patches applied


In [ ]:
%%time
from beir.datasets.data_loader import GenericDataLoader
import re, unicodedata, random, pickle

DATA_PATH = "/content/drive/MyDrive/Datasets/hotpotqa"
TOP_K = 100
RANDOM_SEED = 42
NON_RELEVANT_PAD = 35000

def shared_clean(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

corpus, queries, qrels = GenericDataLoader(data_folder=DATA_PATH).load(split="test")

for doc_id, doc in corpus.items():
    doc["title"] = shared_clean(doc.get("title", ""))
    doc["text"] = shared_clean(doc.get("text", ""))
for q_id, q in queries.items():
    queries[q_id] = shared_clean(q)

# Rebuilding the exact same subset
relevant_doc_ids = set()
for q_id, docs in qrels.items():
    relevant_doc_ids.update(docs.keys())
corpus_subset = {did: corpus[did] for did in relevant_doc_ids if did in corpus}
non_relevant_ids = [did for did in corpus if did not in relevant_doc_ids]
random.seed(RANDOM_SEED)
pad_ids = random.sample(non_relevant_ids, min(NON_RELEVANT_PAD, len(non_relevant_ids)))
for did in pad_ids:
    corpus_subset[did] = corpus[did]
subset_doc_ids = list(corpus_subset.keys())
print(f"Subset rebuilt: {len(corpus_subset):,} docs")

# Loading saved BM25 results
bm25_results = pickle.load(open("/content/drive/MyDrive/IR CW2/bm25_results_hotpotqa.pkl", "rb"))
print(f"BM25 results loaded: {len(bm25_results):,} queries")

# Building qrels + evaluator
qrels_int = {q: {d: int(s) for d, s in docs.items()} for q, docs in qrels.items()}
from pytrec_eval import RelevanceEvaluator
evaluator = RelevanceEvaluator(qrels_int, {"ndcg_cut.10", "recip_rank.10", "recall.100"})
print("Evaluator ready")

  0%|          | 0/5233329 [00:00<?, ?it/s]

Subset rebuilt: 48,783 docs
BM25 results loaded: 7,405 queries
Evaluator ready
CPU times: user 2min 40s, sys: 5.13 s, total: 2min 45s
Wall time: 2min 51s


In [ ]:
colbert_documents = []
colbert_document_ids = []

for doc_id in subset_doc_ids:
    doc = corpus_subset[doc_id]
    full_text = f"{doc['title']} {doc['text']}".strip()
    colbert_documents.append(full_text)
    colbert_document_ids.append(str(doc_id))

print(f"ColBERT docs prepared: {len(colbert_documents):,}")

ColBERT docs prepared: 48,783


In [ ]:
%%time
RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

RAG.index(
    collection=colbert_documents,
    document_ids=colbert_document_ids,
    index_name="hotpotqa_colbert",
    max_document_length=180,
    split_documents=False,
    use_faiss=True
)
print("ColBERT indexing complete")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


artifact.metadata: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

2it [02:52, 86.41s/it]
100%|██████████| 2/2 [00:00<00:00, 144.66it/s]


[Apr 02, 13:15:47] #> Optimizing IVF to store map from centroids to list of pids..
[Apr 02, 13:15:47] #> Building the emb2pid mapping..
[Apr 02, 13:15:47] len(emb2pid) = 3193946


100%|██████████| 16384/16384 [00:00<00:00, 31988.93it/s]

[Apr 02, 13:15:48] #> Saved optimized IVF to .ragatouille/colbert/indexes/hotpotqa_colbert/ivf.pid.pt


Done indexing!
ColBERT indexing complete
CPU times: user 56min 42s, sys: 10.7 s, total: 56min 53s
Wall time: 1h 23s


In [ ]:
q = list(queries.values())[0]
print(f"Query: {q}")
for hit in RAG.search(q, k=5):
    print(f"  [{hit['score']:.4f}] {hit['document_id']}")

Query: Were Scott Derrickson and Ed Wood of the same nationality?
Loading searcher for index hotpotqa_colbert for the first time... This may take a few seconds


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Apr 02, 13:17:27] #> Loading codec...
[Apr 02, 13:17:27] #> Loading IVF...
[Apr 02, 13:17:27] #> Loading doclens...


/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
100%|██████████| 2/2 [00:00<00:00, 468.04it/s]

[Apr 02, 13:17:27] #> Loading codes and residuals...



100%|██████████| 2/2 [00:00<00:00, 12.40it/s]


Searcher loaded!

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: . Were Scott Derrickson and Ed Wood of the same nationality?, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  2020,  3660, 18928,  3385,  1998,  3968,  3536,  1997,
         1996,  2168, 10662,  1029,   102,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

  [17.2344] 10520
  [16.1719] 2816539
  [15.0703] 7530027
  [14.4375] 33453273
  [14.3125] 400595


In [ ]:
%%time
from tqdm import tqdm

colbert_results = {}
for q_id, q_text in tqdm(queries.items(), desc="ColBERT"):
    hits = RAG.search(q_text, k=TOP_K)
    colbert_results[q_id] = {str(h["document_id"]): float(h["score"]) for h in hits}

print(f"ColBERT retrieval done: {len(colbert_results):,} queries")

ColBERT: 100%|██████████| 7405/7405 [04:26<00:00, 27.83it/s]

ColBERT retrieval done: 7,405 queries
CPU times: user 4min 19s, sys: 876 ms, total: 4min 20s
Wall time: 4min 26s


In [ ]:
colbert_eval = evaluator.evaluate(colbert_results)
colbert_ndcg   = [colbert_eval[q]["ndcg_cut_10"] for q in colbert_eval]
colbert_mrr    = [colbert_eval[q]["recip_rank_10"]   for q in colbert_eval]
colbert_recall = [colbert_eval[q]["recall_100"]   for q in colbert_eval]

print("=" * 50)
print("ColBERT Results — HotpotQA")
print("=" * 50)
print(f"nDCG@10:    {sum(colbert_ndcg)/len(colbert_ndcg):.4f}")
print(f"MRR:        {sum(colbert_mrr)/len(colbert_mrr):.4f}")
print(f"Recall@100: {sum(colbert_recall)/len(colbert_recall):.4f}")

import pickle
with open("/content/drive/MyDrive/IR CW2/colbert_results_hotpotqa.pkl", "wb") as f:
    pickle.dump(colbert_results, f)
print("ColBERT results saved to Drive")

ColBERT Results — HotpotQA
nDCG@10:    0.8575
MRR:        0.9770
Recall@100: 0.9356
ColBERT results saved to Drive


## RRF Fusion

In [ ]:
from collections import defaultdict

def rrf_fuse(bm25_res, colbert_res, k=60):
    """Reciprocal Rank Fusion (Cormack et al., 2009).
    RRF(d) = sum_r 1/(k + rank_r(d)), k=60.
    """
    fused = {}
    for qid in set(bm25_res) | set(colbert_res):
        scores = defaultdict(float)
        for rank, (did, _) in enumerate(list(bm25_res.get(qid, {}).items()), 1):
            scores[did] += 1.0 / (k + rank)
        for rank, (did, _) in enumerate(list(colbert_res.get(qid, {}).items()), 1):
            scores[did] += 1.0 / (k + rank)
        fused[qid] = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))
    return fused

rrf_results = rrf_fuse(bm25_results, colbert_results, k=60)
print(f"RRF fusion: {len(rrf_results):,} queries")

RRF fusion: 7,405 queries


In [ ]:
rrf_eval = evaluator.evaluate(rrf_results)
rrf_ndcg   = [rrf_eval[q]["ndcg_cut_10"] for q in rrf_eval]
rrf_mrr    = [rrf_eval[q]["recip_rank_10"]   for q in rrf_eval]
rrf_recall = [rrf_eval[q]["recall_100"]   for q in rrf_eval]

print("=" * 50)
print("RRF Hybrid Results — HotpotQA")
print("=" * 50)
print(f"nDCG@10:    {sum(rrf_ndcg)/len(rrf_ndcg):.4f}")
print(f"MRR:        {sum(rrf_mrr)/len(rrf_mrr):.4f}")
print(f"Recall@100: {sum(rrf_recall)/len(rrf_recall):.4f}")

import pickle
with open("/content/drive/MyDrive/IR CW2/rrf_results_hotpotqa.pkl", "wb") as f:
    pickle.dump(rrf_results, f)

print("RRF results saved to Drive")

RRF Hybrid Results — HotpotQA
nDCG@10:    0.8645
MRR:        0.9709
Recall@100: 0.9490
RRF results saved to Drive


## Final Comparison

In [ ]:
import pickle
from pytrec_eval import RelevanceEvaluator

# Reloading saved results
bm25_results = pickle.load(open("/content/drive/MyDrive/IR CW2/bm25_results_hotpotqa.pkl", "rb"))
colbert_results = pickle.load(open("/content/drive/MyDrive/IR CW2/colbert_results_hotpotqa.pkl", "rb"))
rrf_results = pickle.load(open("/content/drive/MyDrive/IR CW2/rrf_results_hotpotqa.pkl", "rb"))

# Rebuilding evaluator
qrels_int = {q: {d: int(s) for d, s in docs.items()} for q, docs in qrels.items()}
evaluator = RelevanceEvaluator(qrels_int, {"ndcg_cut.10", "recip_rank", "recall.100"})

# Helper: compute MRR@10 manually from ranked results
def compute_mrr_at_10(results, qrels_int):
    mrr_scores = []
    for qid in results:
        relevant = set(d for d, s in qrels_int.get(qid, {}).items() if s > 0)
        ranked_docs = list(results[qid].keys())[:10]
        rr = 0.0
        for rank, doc_id in enumerate(ranked_docs, 1):
            if doc_id in relevant:
                rr = 1.0 / rank
                break
        mrr_scores.append(rr)
    return mrr_scores

# BM25
bm25_eval = evaluator.evaluate(bm25_results)
bm25_ndcg = [bm25_eval[q]["ndcg_cut_10"] for q in bm25_eval]
bm25_mrr = compute_mrr_at_10(bm25_results, qrels_int)
bm25_recall = [bm25_eval[q]["recall_100"] for q in bm25_eval]

# ColBERT
colbert_eval = evaluator.evaluate(colbert_results)
colbert_ndcg = [colbert_eval[q]["ndcg_cut_10"] for q in colbert_eval]
colbert_mrr = compute_mrr_at_10(colbert_results, qrels_int)
colbert_recall = [colbert_eval[q]["recall_100"] for q in colbert_eval]

# RRF
rrf_eval = evaluator.evaluate(rrf_results)
rrf_ndcg = [rrf_eval[q]["ndcg_cut_10"] for q in rrf_eval]
rrf_mrr = compute_mrr_at_10(rrf_results, qrels_int)
rrf_recall = [rrf_eval[q]["recall_100"] for q in rrf_eval]

print("All metric summaries rebuilt")

All metric summaries rebuilt


In [ ]:
print()
print("=" * 62)
print(f"{'Method':<12} {'nDCG@10':>10} {'MRR':>10} {'Recall@100':>12}")
print("=" * 62)
print(f"{'BM25':<12} {sum(bm25_ndcg)/len(bm25_ndcg):>10.4f} {sum(bm25_mrr)/len(bm25_mrr):>10.4f} {sum(bm25_recall)/len(bm25_recall):>12.4f}")
print(f"{'ColBERT':<12} {sum(colbert_ndcg)/len(colbert_ndcg):>10.4f} {sum(colbert_mrr)/len(colbert_mrr):>10.4f} {sum(colbert_recall)/len(colbert_recall):>12.4f}")
print(f"{'RRF Hybrid':<12} {sum(rrf_ndcg)/len(rrf_ndcg):>10.4f} {sum(rrf_mrr)/len(rrf_mrr):>10.4f} {sum(rrf_recall)/len(rrf_recall):>12.4f}")
print("=" * 62)
print(f"\nSubset: {len(corpus_subset):,} docs | Queries: {len(queries):,} | TOP_K: {TOP_K}")


Method          nDCG@10        MRR   Recall@100
BM25             0.8120     0.9349       0.9211
ColBERT          0.8575     0.9768       0.9356
RRF Hybrid       0.8645     0.9691       0.9490

Subset: 48,783 docs | Queries: 7,405 | TOP_K: 100


In [ ]:
import json
import pickle
from pathlib import Path

TOP_N_UI = 10

OUT_PATH = "/content/drive/MyDrive/IR CW2/hotpotqa_demo_results.json"

def safe_text(text):
    if not isinstance(text, str):
        return ""
    return " ".join(text.split())

def make_snippet(doc, max_chars=220):
    title = safe_text(doc.get("title", ""))
    text = safe_text(doc.get("text", ""))
    full = f"{title}. {text}" if title and text else (title or text)
    return full[:max_chars].rstrip() + ("..." if len(full) > max_chars else "")

def build_method_results(method_results, query_id, corpus_lookup, top_n=10):
    ranked_items = list(method_results.get(query_id, {}).items())[:top_n]
    output = []
    for rank, (doc_id, score) in enumerate(ranked_items, start=1):
        doc_id_str = str(doc_id)
        doc = corpus_lookup.get(doc_id) or corpus_lookup.get(doc_id_str) or {}
        output.append({
            "rank": rank,
            "doc_id": doc_id_str,
            "title": safe_text(doc.get("title", "")),
            "score": float(score),
            "snippet": make_snippet(doc)
        })
    return output

bm25_path = "/content/drive/MyDrive/IR CW2/bm25_results_hotpotqa.pkl"
colbert_path = "/content/drive/MyDrive/IR CW2/colbert_results_hotpotqa.pkl"
rrf_path = "/content/drive/MyDrive/IR CW2/rrf_results_hotpotqa.pkl"

bm25_results = pickle.load(open(bm25_path, "rb"))
colbert_results = pickle.load(open(colbert_path, "rb"))
rrf_results = pickle.load(open(rrf_path, "rb"))

demo_data = {}
for query_id, query_text in queries.items():
    demo_data[query_text] = {
        "query_id": str(query_id),
        "bm25": build_method_results(bm25_results, query_id, corpus_subset, top_n=TOP_N_UI),
        "colbert": build_method_results(colbert_results, query_id, corpus_subset, top_n=TOP_N_UI),
        "rrf": build_method_results(rrf_results, query_id, corpus_subset, top_n=TOP_N_UI),
    }

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(demo_data, f, indent=2, ensure_ascii=False)

print(f"Saved demo JSON to: {OUT_PATH}")
print(f"Total queries saved: {len(demo_data):,}")
print(f"Results per method per query: {TOP_N_UI}")

Saved demo JSON to: /content/drive/MyDrive/IR CW2/hotpotqa_demo_results.json
Total queries saved: 7,405
Results per method per query: 10
